In [ ]:
import os
import re
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv


load_dotenv()

/data/oussama/NYU/ve/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
os.environ["HF_HOME"] = "/data/oussama/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/data/oussama/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/data/oussama/hf_cache/transformers"

# Gemma

In [9]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM


def run_gemma4_local_on_questions(
    input_csv,
    output_csv,
    prompt_col="Prompt",
    model_id="google/gemma-4-12B-it",
    question_id_col=None,
    gold_col=None,
    level_col="level",
    medical_specialty_col="medical_specialty",
    max_retries=5,
    save_every=1,
    max_new_tokens=10,
):
    """
    Run Gemma 4 locally using Transformers and save predictions.

    Output CSV columns:
    - question_id
    - gold
    - prediction
    - raw_output
    - correct
    - level
    - medical_specialty
    - model
    - stop_reason
    - refusal_category
    - refusal_type
    """

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    print(f"Loading local Gemma model: {model_id}")

    processor = AutoProcessor.from_pretrained(model_id)

    gemma_model = AutoModelForMultimodalLM.from_pretrained(
        model_id,
        dtype="auto",
        cache_dir="/data/oussama/hf_cache",
        device_map="auto",
    )

    gemma_model.eval()

    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    if prompt_col not in df.columns:
        raise ValueError(f"Missing prompt column: {prompt_col}")

    if gold_col is None:
        raise ValueError(
            "Could not find the gold answer column. "
            "Please pass gold_col='your_column_name'."
        )

    output_columns = [
        "question_id",
        "gold",
        "prediction",
        "raw_output",
        "correct",
        "level",
        "medical_specialty",
        "model",
        "stop_reason",
        "refusal_category",
        "refusal_type",
    ]

    def clean_value(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def normalize_gold(value):
        text = clean_value(value).upper()
        match = re.search(r"\b([A-F])\b", text)
        return match.group(1) if match else text[:1]

    def extract_prediction(raw_output, allowed_letters=None):
        raw_output = clean_value(raw_output).upper()

        if allowed_letters:
            allowed_letters = [x.upper() for x in allowed_letters]
        else:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        if raw_output in allowed_letters:
            return raw_output

        pattern = r"\b(" + "|".join(map(re.escape, allowed_letters)) + r")\b"
        match = re.search(pattern, raw_output)
        if match:
            return match.group(1)

        if raw_output and raw_output[0] in allowed_letters:
            return raw_output[0]

        return ""

    def parse_gemma_output(decoded_text):
        """
        Robustly parse Gemma 4 output.
        processor.parse_response may return different structures depending on version.
        """

        text = decoded_text

        try:
            parsed = processor.parse_response(decoded_text)

            if isinstance(parsed, str):
                text = parsed

            elif isinstance(parsed, dict):
                text = (
                    parsed.get("content")
                    or parsed.get("answer")
                    or parsed.get("text")
                    or str(parsed)
                )

            elif isinstance(parsed, list) and len(parsed) > 0:
                first = parsed[0]
                if isinstance(first, dict):
                    text = (
                        first.get("content")
                        or first.get("text")
                        or str(first)
                    )
                else:
                    text = str(first)

            else:
                text = str(parsed)

        except Exception:
            text = decoded_text

        text = clean_value(text)

        # Remove common special tokens if they appear.
        text = text.replace("<turn|>", " ")
        text = text.replace("<end_of_turn>", " ")
        text = text.replace("<start_of_turn>", " ")
        text = text.strip()

        return text

    def call_model(prompt, allowed_letters=None, max_retries=5):
        """
        Calls Gemma 4 locally.

        Returns:
        {
            "raw_output": str,
            "stop_reason": str,
            "refusal_category": None,
            "refusal_type": None,
        }
        """

        if allowed_letters is None:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        allowed_text = ", ".join(allowed_letters)

        system_prompt = (
            "You are evaluating multiple-choice questions from a medical benchmark. "
            "Select the single best answer option. "
            "Return only one uppercase letter. "
            "Do not explain."
        )

        user_prompt = (
            f"{prompt}\n\n"
            f"Allowed options: {allowed_text}\n"
            "Return exactly one uppercase letter from the allowed options. "
            "No explanation."
        )

        messages = [
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ]

        last_error = None

        for attempt in range(1, max_retries + 1):
            try:
                inputs = processor.apply_chat_template(
                    messages,
                    tokenize=True,
                    return_dict=True,
                    return_tensors="pt",
                    add_generation_prompt=True,
                    enable_thinking=False,
                ).to(gemma_model.device)

                input_len = inputs["input_ids"].shape[-1]

                with torch.inference_mode():
                    outputs = gemma_model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        pad_token_id=processor.tokenizer.eos_token_id,
                    )

                generated_tokens = outputs[0][input_len:]

                decoded = processor.decode(
                    generated_tokens,
                    skip_special_tokens=False,
                )

                raw_output = parse_gemma_output(decoded)

                if raw_output:
                    return {
                        "raw_output": raw_output,
                        "stop_reason": "local_generate",
                        "refusal_category": None,
                        "refusal_type": None,
                    }

                print("Empty local Gemma response:")
                print("decoded:", repr(decoded))

                raise RuntimeError("Gemma returned empty text output.")

            except Exception as e:
                last_error = e

                if attempt == max_retries:
                    break

                wait_time = min(2 ** attempt, 60)
                print(
                    f"Local Gemma error attempt {attempt}/{max_retries}: {repr(e)}. "
                    f"Retrying in {wait_time}s..."
                )
                time.sleep(wait_time)

        raise RuntimeError(
            f"Failed after {max_retries} retries. Last error: {repr(last_error)}"
        )

    # Read already processed question IDs if results CSV exists
    processed_ids = set()

    if output_csv.exists():
        old_results = pd.read_csv(output_csv, encoding="utf-8-sig")

        if "question_id" in old_results.columns:
            processed_ids = set(old_results["question_id"].astype(str))
        elif "Question_id" in old_results.columns:
            processed_ids = set(old_results["Question_id"].astype(str))

        for col in output_columns:
            if col not in old_results.columns:
                old_results[col] = ""

        old_results = old_results[output_columns]
        results = old_results.to_dict("records")
    else:
        old_results = pd.DataFrame(columns=output_columns)
        results = []

    print(f"Already processed: {len(processed_ids)} questions")

    newly_processed = 0

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        question_id = (
            clean_value(row[question_id_col])
            if question_id_col is not None
            else str(idx)
        )

        if str(question_id) in processed_ids:
            continue

        gold = normalize_gold(row[gold_col])

        group = clean_value(row["Group"]).upper() if "Group" in df.columns else "ABCDEF"

        allowed_letters = [
            letter for letter in group
            if letter in ["A", "B", "C", "D", "E", "F"]
        ]

        if not allowed_letters:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        prompt = clean_value(row[prompt_col])

        call_result = call_model(
            prompt=prompt,
            allowed_letters=allowed_letters,
            max_retries=max_retries,
        )

        raw_output = call_result["raw_output"]
        stop_reason = call_result["stop_reason"]
        refusal_category = call_result["refusal_category"]
        refusal_type = call_result["refusal_type"]

        prediction = extract_prediction(raw_output, allowed_letters)
        correct = int(prediction == gold)

        result = {
            "question_id": question_id,
            "gold": gold,
            "prediction": prediction,
            "raw_output": raw_output,
            "correct": correct,
            "level": clean_value(row[level_col]) if level_col in df.columns else "",
            "medical_specialty": (
                clean_value(row[medical_specialty_col])
                if medical_specialty_col in df.columns
                else ""
            ),
            "model": model_id,
            "stop_reason": stop_reason,
            "refusal_category": refusal_category,
            "refusal_type": refusal_type,
        }

        results.append(result)
        processed_ids.add(str(question_id))
        newly_processed += 1

        if save_every and newly_processed % save_every == 0:
            pd.DataFrame(results, columns=output_columns).to_csv(
                output_csv,
                index=False,
                encoding="utf-8-sig",
            )

    results_df = pd.DataFrame(results, columns=output_columns)

    results_df.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return results_df

In [ ]:
# gemma-4-31B-it 
results = run_gemma4_local_on_questions(
    input_csv="data/cleaned/medarabench_test_with_prompts.csv",
    output_csv="results/gemma-4-12B-it_results.csv",
    prompt_col="Prompt",
    model_id="google/gemma-4-12B-it",
    question_id_col="Question Number",
    gold_col="Correct Answer",
    level_col="Level",
    medical_specialty_col="Medical Specialty",
    save_every=1,
)

Loading local Gemma model: google/gemma-4-12B-it


Loading weights: 100%|██████████| 677/677 [00:00<00:00, 4173.11it/s]


Already processed: 0 questions


100%|██████████| 25/25 [10:42<00:00, 25.69s/it]
